In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.types import *

source_path = '/Volumes/workspace/madstage/stage/stageArea'
files = dbutils.fs.ls(source_path)

c_schema = StructType([
    StructField('customer_id',IntegerType(),False),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType()),
    StructField('load_date',StringType()),
])

a_schema = StructType([
    StructField('account_id',IntegerType()),
    StructField('customer_id',IntegerType(),False),
    StructField('account_type',StringType()),
    StructField('branch_city',StringType()),
    StructField('status',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType())
])

t_schema = StructType(
    [
        StructField('transaction_id',IntegerType()),
        StructField('account_id',IntegerType()),
        StructField('transaction_type',StringType()),
        StructField('amount',IntegerType()),
        StructField('transaction_date',StringType())
    ]
)

for file in files:
    if file.name.startswith('customer'):
        print('customer file found')
        customers = (spark.read.format('csv').option('header','true').schema(c_schema).load(file.path))
    elif file.name.startswith('account'):
        print('account file found!')
        account = (spark.read.format('csv').option('header','true').schema(a_schema).load(file.path))
    elif file.name.startswith('transaction'):
        print('transaction file found!')
        trans = (spark.read.format('csv').option('header','true').schema(t_schema).load(file.path))

# drop nulls
customers = customers.dropna(subset='customer_id')
account = account.dropna(subset=('customer_id'))
trans = trans.dropna(subset=('transaction_id'))

# drop duplicates 11
customers = customers.dropDuplicates()
account = account.dropDuplicates()
trans = trans.dropDuplicates()

# nullhandling
customers = customers.fillna({
    'customer_name':'Unknown',
    'city':'Unknown',
    'customer_segment':'Unknown',
    'email':'Unknown',
    'phone':'Unknown'
})

account = account.fillna({
    'account_type':'Unknown',
    'branch_city':'Unknown',
    'status':'Unknown'
})

trans = trans.fillna({
    'transaction_type':'Unknown'
})

# standardization
phone_regex = r'[^0-9]'
customers = customers.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.lower(f.col('email'))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,'')).withColumn('is_current',f.initcap(f.trim('is_current'))).withColumn('start_date',f.trim('start_date')).withColumn('end_date',f.trim('end_date')).orderBy(f.col('customer_id'))

account = account.withColumn('account_type',f.initcap(f.trim(f.col('account_type')))).withColumn('branch_city',f.initcap(f.trim('branch_city'))).withColumn('status',f.initcap(f.trim('status'))).withColumn('is_current',f.initcap(f.trim('is_current'))).withColumn('start_date',f.trim('start_date')).withColumn('end_date',f.trim('end_date')).orderBy(f.col('customer_id'))

trans = trans.withColumn('transaction_type',f.initcap(f.trim(f.col('transaction_type')))).orderBy(f.col('account_id')).withColumn('transaction_date',f.trim(f.col('transaction_date')))

# data validation 
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation = (f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone'))==10)
amount_validation = (f.col('amount') >= 0)

customers = customers.withColumn('data_validation',f.when((email_validation) & (phone_validation),'Pass').otherwise('Fail'))
account = account.withColumn('data_validation',f.when((f.col('status')== 'Active'),'Pass').otherwise('Fail'))
trans = trans.withColumn('data_validation',f.when((amount_validation),'Pass').otherwise('Fail'))

customers = customers.filter(f.col('data_validation')=='Pass')
account = account.filter(f.col('data_validation')=='Pass')
trans = trans.filter(f.col('data_validation')=='Pass')

# aduit columns
customers = customers.select('customer_id','customer_name','city','customer_segment','email','phone','is_current')
account = account.select('account_id','customer_id','account_type','branch_city','status','is_current')
customers = customers.withColumn('updated_customer_dt',f.lit('9999-12-31').cast('string')).withColumn('created_customer_dt',f.current_date().cast('string'))
account = account.withColumn('updated_account_dt',f.lit('9999-12-31').cast('string')).withColumn('created_account_dt',f.current_date().cast('string'))
trans = trans.drop('data_validation').withColumn('is_current',f.lit('Y'))

# customers.show()
# account.show()
# trans.show()

# customers.printSchema()
# account.printSchema()
# trans.printSchema()

# write into delta tables
customers.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madstage.customers')
account.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madstage.accounts')
trans.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.madstage.transactions')

print('Intial load completed')